In [ ]:
import torch
from torch import nn
from tqdm import tqdm

In [ ]:
BLOCK_SIZE = 256
N_EMBED = 384
N_HEADS = 6
DROPOUT = 0.2
N_LAYERS = 6

TRAIN_SPLIT = 0.9
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
MAX_ITERS = 5000
EVAL_ITERS = 200

RANDOM_SEED = 1337
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
with open("datasets/shakespeare.txt", "r", encoding="utf-8") as file:
    text = file.read()

# print("Length of the text:", len(text))

# chars = sorted(list(set(text)))
# print("Unique characters:", chars)

# char_to_idx = {ch: i for i, ch in enumerate(chars)}
# idx_to_char = {i: ch for i, ch in enumerate(chars)}
# encode = lambda s: [char_to_idx[c] for c in s]
# decode = lambda l: ''.join([idx_to_char[i] for i in l])

# data = torch.tensor(encode(text), dtype=torch.long)
# print("Data tensor shape:", data.shape)
# print("First 100 characters of the data tensor:", data[:100])

# load tokenizer from json
import tokenizers
tokenizer = tokenizers.Tokenizer.from_file("tokenizer.json")

data = torch.tensor(tokenizer.encode(text).ids, dtype=torch.long)
print("Data tensor shape:", data.shape)
print("First 100 characters of the data tensor:", data[:100])

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"Context: {context.tolist()} -> Target: {target.item()}")

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print("Inputs (x):", xb.shape, xb)
print("Targets (y):", yb.shape, yb)

for b in range(BATCH_SIZE):
    for t in range(BLOCK_SIZE):
        context = xb[b, :t + 1]
        target = yb[b, t]
        print(f"Batch {b}, Time {t}: Context: {context.tolist()} -> Target: {target.item()}")

In [ ]:
class HeadSelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBED, head_size, bias=False)
        self.query = nn.Linear(N_EMBED, head_size, bias=False)
        self.value = nn.Linear(N_EMBED, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        # Register_buffer is used to store a tensor that is not a parameter but should be part of the module's state.
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        # Compute attention scores
        weights = q @ k.transpose(-2, -1) * C**-0.5 # (B, T, T)
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Apply causal mask
        weights = torch.softmax(weights, dim=-1) # Normalize to get probabilities
        weights = self.dropout(weights) # Apply dropout to attention weights

        out = weights @ v # (B, T, head_size)
        return out
    
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([HeadSelfAttention(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(N_EMBED, N_EMBED) # Project concatenated heads back to embedding size
        self.dropout = nn.Dropout(DROPOUT) 

    def forward(self, x):
        out =  torch.cat([h(x) for h in self.heads], dim=-1) # Concatenate outputs from all heads
        return self.dropout(self.proj(out))
    
class FeedForward(nn.Module):
    def __init__(self, connections, neurons):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(connections, neurons),
            nn.ReLU(),
            nn.Linear(neurons, connections),
            nn.Dropout(DROPOUT)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.self_attention = MultiHeadSelfAttention(num_heads=n_heads, head_size=head_size)
        self.ffwd = FeedForward(n_embed, 4 * n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.self_attention(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embed):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) # Embedding for characters
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, n_embed) # Embedding for positions
        self.blocks = nn.Sequential(
            *[Block(n_embed, N_HEADS) for _ in range(N_LAYERS)],
            nn.LayerNorm(n_embed)
        )
        self.lm_head = nn.Linear(n_embed, vocab_size) # Project embeddings to vocabulary

    def forward(self, idx, targets=None):
        token_embeds = self.token_embedding_table(idx) # (batch size, block size, n_embed)
        pos_embeds = self.position_embedding_table(torch.arange(idx.size(1), device=idx.device)) # (block size, n_embed)

        x = token_embeds + pos_embeds # Combine token and position embeddings
        x = self.blocks(x) # Apply multiple blocks of self-attention and feedforward layers
        logits = self.lm_head(x) # (batch size, block size, vocab size)
        
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        logits = logits.view(B * T, C)
        targets = targets.view(B * T)

        loss = nn.functional.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]  # crop context to the last BLOCK_SIZE tokens
            # get the predictions
            logits, loss = self(idx_cond)
            # focus on last time step
            logits = logits[:, -1, :]
            probabilities = nn.functional.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probabilities, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

torch.manual_seed(RANDOM_SEED)
model = BigramLanguageModel(vocab_size=tokenizer.get_vocab_size(), n_embed=N_EMBED)
model = model.to(device)
logits, loss = model(xb, yb)

print(loss)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_sequence = model.generate(context, max_new_tokens=100)
print("Generated sequence:", tokenizer.decode(generated_sequence[0].tolist()))

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

history = {'train': [], 'val': []}
pbar = tqdm(range(MAX_ITERS))
for step in pbar:
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % EVAL_ITERS == 0:
        losses = estimate_loss()
        history['train'].append(losses['train'])
        history['val'].append(losses['val'])
    pbar.set_description(f"Step {step}: Train Loss: {loss.item():.4f}, Val Loss: {losses['val']:.4f}")

| Version              | Train loss | Train time [min] | Test loss | Test time [s] | Qualitative result |
|----------------------|:------------:|:------------------:|:-----------:|:---------------:|--------------------|
| Character tokenizer  | 1.195 |          45        |     1.490      |       18.6        |          Decent, some word errors, little sense though          |
| Learnt BPE tokenizer |     0.810       |      49            |    6.384       |     16.4          |         More sensical, less invented words           |

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_sequence = model.generate(context, max_new_tokens=1000)
print("Generated sequence:", tokenizer.decode(generated_sequence[0].tolist()))

In [ ]:
# bag of words (averaging)
B, T, C = 4, 8, 2
torch.manual_seed(RANDOM_SEED)
x = torch.rand(B, T, C)
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]
        xbow[b, t] = torch.mean(xprev, dim=0)

# second way
weights = torch.tril(torch.ones(T, T))
weights = weights / weights.sum(1, keepdim=True)
xbow2 = weights @ x

# third way
tril = torch.tril(torch.ones(T, T))
weights = torch.zeros((T, T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = torch.nn.functional.softmax(weights, dim=-1)
xbow3 = weights @ x

# 4th, self attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
v = value(x)
weights = q @ k.transpose(-2, -1) * head_size ** -0.5

tril = torch.tril(torch.ones(T, T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = torch.nn.functional.softmax(weights, dim=-1)
xbow4 = weights @ v